In [37]:
import sys
sys.path.append('.')
from pathlib import Path
import ast
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import wfdb
import neurokit2 as nk
from scipy.signal import resample
import math

# -------------------------
# Config
# -------------------------
BEAT_LENGTH  = 300
MAX_BEATS    = 35
IN_CHANNELS  = 12
BEAT_DIM     = IN_CHANNELS * BEAT_LENGTH

D_MODEL      = 256
NUM_HEADS    = 8
NUM_LAYERS   = 4
MLP_RATIO    = 4
DROPOUT      = 0.1

In [38]:
class FixedCNNTokenizer(nn.Module):
    def __init__(self, in_channels=12, d_model=256, patch_size=300):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv1d(
            in_channels=in_channels,
            out_channels=d_model,
            kernel_size=patch_size,
            stride=patch_size,
            bias=True,
        )

    def forward(self, x):
        z = self.proj(x)
        z = z.transpose(1, 2)
        return z


class SinCosPositionalEncoding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        assert d_model % 2 == 0
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) *
            (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(dtype=x.dtype, device=x.device)


class TransformerEncoder(nn.Module):
    def __init__(self, d_model=256, num_heads=8, num_layers=4, mlp_ratio=4, dropout=0.1):
        super().__init__()
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_model * mlp_ratio,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

    def forward(self, x, src_key_padding_mask=None):
        return self.encoder(x, src_key_padding_mask=src_key_padding_mask)


class ECGMaskedSSLBeat(nn.Module):
    def __init__(self, in_channels=12, d_model=256, beat_length=300,
                 num_heads=8, num_layers=4, mlp_ratio=4, dropout=0.1, max_beats=35):
        super().__init__()
        self.beat_dim   = in_channels * beat_length
        self.tokenizer  = FixedCNNTokenizer(in_channels, d_model, beat_length)
        self.posenc     = SinCosPositionalEncoding(max_beats, d_model)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        self.encoder    = TransformerEncoder(d_model, num_heads, num_layers, mlp_ratio, dropout)
        self.pred_head  = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, self.beat_dim),
        )

    def forward(self, beats, padding_mask=None, mask=None, mask_ratio=0.50, span_len=1):
        B, N, C, T = beats.shape
        tokens = self.tokenizer(beats.view(B * N, C, T))
        tokens = tokens.squeeze(1).view(B, N, -1)
        B, N, D = tokens.shape

        target_patches = beats.reshape(B, N, self.beat_dim)

        if mask is None:
            mask = torch.zeros(B, N, dtype=torch.bool, device=tokens.device)
        if padding_mask is not None:
            mask = mask & ~padding_mask

        mask_token    = self.mask_token.expand(B, N, D)
        masked_tokens = torch.where(mask.unsqueeze(-1), mask_token, tokens)
        masked_tokens = self.posenc(masked_tokens)
        encoded       = self.encoder(masked_tokens, src_key_padding_mask=padding_mask)
        pred_patches  = self.pred_head(encoded)

        if padding_mask is not None:
            real   = (~padding_mask).unsqueeze(-1).float()
            pooled = (encoded * real).sum(dim=1) / real.sum(dim=1).clamp(min=1)
        else:
            pooled = encoded.mean(dim=1)

        return {
            "pred_patches":   pred_patches,
            "target_patches": target_patches,
            "mask":           mask,
            "encoded":        encoded,
            "pooled":         pooled,
        }

In [39]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

checkpoint_path = Path("checkpoints_tok1_ssl/best.pt")
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

pretrained_model = ECGMaskedSSLBeat(
    in_channels=IN_CHANNELS,
    d_model=D_MODEL,
    beat_length=BEAT_LENGTH,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    mlp_ratio=MLP_RATIO,
    dropout=DROPOUT,
    max_beats=MAX_BEATS,
).to(device)

state_dict = checkpoint["model_state_dict"]
state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
pretrained_model.load_state_dict(state_dict)
pretrained_model.eval()
print("Loaded checkpoint from:", checkpoint_path)

Loaded checkpoint from: checkpoints_tok1_ssl/best.pt


/tmp/ipykernel_2237655/2806447164.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


In [40]:
class PTBXLBeatClassifier(nn.Module):
    def __init__(self, pretrained_model, feature_dim=256, num_classes=1):
        super().__init__()
        self.pretrained_model = pretrained_model
        self.classifier = nn.Linear(feature_dim, num_classes)

    def forward(self, beats, padding_mask):
        out    = self.pretrained_model(beats, padding_mask=padding_mask)
        pooled = out["pooled"]
        return self.classifier(pooled)


model = PTBXLBeatClassifier(pretrained_model, num_classes=5).to(device)
print(model)

PTBXLBeatClassifier(
  (pretrained_model): ECGMaskedSSLBeat(
    (tokenizer): FixedCNNTokenizer(
      (proj): Conv1d(12, 256, kernel_size=(300,), stride=(300,))
    )
    (posenc): SinCosPositionalEncoding()
    (encoder): TransformerEncoder(
      (encoder): TransformerEncoder(
        (layers): ModuleList(
          (0-3): 4 x TransformerEncoderLayer(
            (self_attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
            )
            (linear1): Linear(in_features=256, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
            (linear2): Linear(in_features=1024, out_features=256, bias=True)
            (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
            (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
            (dropout1): Dropout(p=0.1, inplace=False)
            (dropout2): Dropout(p=0.1, inplace=False)
          )
   

In [41]:
ptbxl_base = Path("/data/rohit/PTB-XL")
label_df   = pd.read_csv(ptbxl_base / "ptbxl_database.csv")
scp_df     = pd.read_csv(ptbxl_base / "scp_statements.csv", index_col=0)

label_df["scp_codes"] = label_df["scp_codes"].apply(ast.literal_eval)

# -------------------------
# Build superclass mapping from scp_statements.csv
# -------------------------
SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

# map each SCP code to its diagnostic superclass
scp_to_superclass = {}
for scp_code, row in scp_df.iterrows():
    if row["diagnostic_class"] in SUPERCLASSES:
        scp_to_superclass[scp_code] = row["diagnostic_class"]

def get_superclass_labels(scp_codes_dict):
    """Returns a 5-dim binary vector [NORM, MI, STTC, CD, HYP]"""
    labels = np.zeros(len(SUPERCLASSES), dtype=np.float32)
    for scp_code, likelihood in scp_codes_dict.items():
        if scp_code in scp_to_superclass: # no likelihood filter
            superclass = scp_to_superclass[scp_code]
            idx = SUPERCLASSES.index(superclass)
            labels[idx] = 1.0
    return labels

label_df["target"] = label_df["scp_codes"].apply(get_superclass_labels)

# drop records with no superclass label
label_df = label_df[label_df["target"].apply(lambda x: x.sum() > 0)].copy()

# paths
label_df["ecg_path"] = label_df.apply(lambda row: ptbxl_base / row["filename_hr"], axis=1)

def files_exist(row):
    base = Path(row["ecg_path"])
    return Path(str(base) + ".hea").exists() and Path(str(base) + ".dat").exists()

label_df["file_exists"] = label_df.apply(files_exist, axis=1)
label_df = label_df[label_df["file_exists"]].copy()

# -------------------------
# Official PTB-XL splits
# -------------------------
train_df = label_df[label_df["strat_fold"] <= 8].copy()
val_df   = label_df[label_df["strat_fold"] == 9].copy()
test_df  = label_df[label_df["strat_fold"] == 10].copy()

print(f"Train: {len(train_df)}")
print(f"Val:   {len(val_df)}")
print(f"Test:  {len(test_df)}")
print(f"Superclasses: {SUPERCLASSES}")
for i, sc in enumerate(SUPERCLASSES):
    n = label_df["target"].apply(lambda x: x[i]).sum()
    print(f"  {sc}: {int(n)}")

Train: 17111
Val:   2156
Test:  2163
Superclasses: ['NORM', 'MI', 'STTC', 'CD', 'HYP']
  NORM: 9528
  MI: 5486
  STTC: 5250
  CD: 4907
  HYP: 2655


In [42]:
import wfdb
import neurokit2 as nk
import numpy as np
import warnings
from pathlib import Path

ptbxl_base = Path("/data/rohit/PTB-XL")

# use the already-built train_df from the PTB-XL downstream notebook
# if running standalone, rebuild it first

print("Computing beat stats on PTB-XL training set (this may take a few minutes)...")

beat_counts = []
for _, row in train_df.iterrows():
    record = wfdb.rdrecord(str(row["ecg_path"]))
    x = record.p_signal.astype(np.float32).T   # (12, 5000)

    x = np.clip(x, -5, 5)
    mean = x.mean(axis=1, keepdims=True)
    std  = x.std(axis=1, keepdims=True)
    x    = (x - mean) / np.clip(std, 1e-4, None)

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            _, info = nk.ecg_peaks(x[1], sampling_rate=500)
        r_peaks = np.array(info["ECG_R_Peaks"])
        n_beats = max(0, len(r_peaks) - 1)
    except Exception:
        n_beats = 0

    beat_counts.append(n_beats)

beat_counts = np.array(beat_counts)
print(f"\nBeat count statistics (PTB-XL training set, N={len(beat_counts)}):")
print(f"  Min     : {beat_counts.min()}")
print(f"  Max     : {beat_counts.max()}")
print(f"  Mean    : {beat_counts.mean():.1f}")
print(f"  Median  : {np.median(beat_counts):.1f}")
print(f"  5th pct : {np.percentile(beat_counts, 5):.1f}")
print(f"  25th pct: {np.percentile(beat_counts, 25):.1f}")
print(f"  75th pct: {np.percentile(beat_counts, 75):.1f}")
print(f"  90th pct: {np.percentile(beat_counts, 90):.1f}")
print(f"  95th pct: {np.percentile(beat_counts, 95):.1f}")
print(f"  99th pct: {np.percentile(beat_counts, 99):.1f}")
print(f"\nCurrent MAX_BEATS = {MAX_BEATS}")
pct_covered = (beat_counts <= MAX_BEATS).mean() * 100
print(f"Records covered by MAX_BEATS: {pct_covered:.1f}%")

Computing beat stats on PTB-XL training set (this may take a few minutes)...

Beat count statistics (PTB-XL training set, N=17111):
  Min     : 0
  Max     : 27
  Mean    : 11.2
  Median  : 11.0
  5th pct : 8.0
  25th pct: 10.0
  75th pct: 12.0
  90th pct: 14.0
  95th pct: 16.0
  99th pct: 19.0

Current MAX_BEATS = 35
Records covered by MAX_BEATS: 100.0%


In [43]:
def detect_r_peaks(lead_ii, sampling_rate=500):
    import warnings
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            _, info = nk.ecg_peaks(lead_ii, sampling_rate=sampling_rate)
        return np.array(info["ECG_R_Peaks"])
    except Exception:
        return np.array([])


def extract_beats(x, beat_length=300, sampling_rate=500, min_beats=3):
    lead_ii = x[1]
    r_peaks = detect_r_peaks(lead_ii, sampling_rate=sampling_rate)

    if len(r_peaks) < 2:
        return None

    beats = []
    for i in range(len(r_peaks) - 1):
        start = r_peaks[i]
        end   = r_peaks[i + 1]
        beat  = x[:, start:end]
        if beat.shape[1] < 10:
            continue
        resampled = np.zeros((12, beat_length), dtype=np.float32)
        for c in range(12):
            resampled[c] = resample(beat[c], beat_length)
        beats.append(resampled)

    if len(beats) < min_beats:
        return None

    return np.stack(beats, axis=0)   # (N, 12, beat_length)


class PTBXLBeatDataset(Dataset):
    def __init__(self, df, beat_length=300):
        self.df          = df.reset_index(drop=True)
        self.beat_length = beat_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row         = self.df.iloc[idx]
        record_path = str(row["ecg_path"])

        record = wfdb.rdrecord(record_path)
        x      = record.p_signal.astype(np.float32).T   # (12, 5000)

        x = np.clip(x, -5, 5)
        mean = x.mean(axis=1, keepdims=True)
        std  = x.std(axis=1, keepdims=True)
        x    = (x - mean) / np.clip(std, 1e-4, None)

        beats = extract_beats(x, beat_length=self.beat_length)
        if beats is None:
            # fallback: return neighbor
            return self.__getitem__((idx + 1) % len(self))

        y = torch.tensor(row["target"], dtype=torch.float32)   # (5,)
        return torch.from_numpy(beats), y   # (N, 12, beat_length), (5,)


def beat_collate_fn(batch):
    beats_list, labels = zip(*batch)
    num_beats = [b.shape[0] for b in beats_list]
    max_n     = max(num_beats)
    B         = len(beats_list)

    padded_beats = torch.zeros(B, max_n, IN_CHANNELS, BEAT_LENGTH)
    padding_mask = torch.ones(B, max_n, dtype=torch.bool)

    for i, beats in enumerate(beats_list):
        n = beats.shape[0]
        padded_beats[i, :n] = beats
        padding_mask[i, :n] = False

    return padded_beats, padding_mask, torch.stack(labels)

In [44]:
train_dataset = PTBXLBeatDataset(train_df, beat_length=BEAT_LENGTH)
val_dataset   = PTBXLBeatDataset(val_df,   beat_length=BEAT_LENGTH)
test_dataset  = PTBXLBeatDataset(test_df,  beat_length=BEAT_LENGTH)

batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=8, collate_fn=beat_collate_fn,
                          persistent_workers=True, prefetch_factor=2)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                          num_workers=8, collate_fn=beat_collate_fn,
                          persistent_workers=True, prefetch_factor=2)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                          num_workers=8, collate_fn=beat_collate_fn,
                          persistent_workers=True, prefetch_factor=2)

print("Dataloaders ready")

Dataloaders ready


In [45]:
from sklearn.metrics import roc_auc_score, average_precision_score

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for beats, padding_mask, y in loader:
        beats        = beats.to(device)
        padding_mask = padding_mask.to(device)
        y            = y.to(device)   # (B, 5)

        optimizer.zero_grad()
        logits = model(beats, padding_mask)
        loss   = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_probs  = []
    all_labels = []

    with torch.no_grad():
        for beats, padding_mask, y in loader:
            beats        = beats.to(device)
            padding_mask = padding_mask.to(device)
            y            = y.to(device)   # (B, 5)

            logits = model(beats, padding_mask)
            loss   = criterion(logits, y)
            probs  = torch.sigmoid(logits)

            total_loss  += loss.item()
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    all_probs  = np.array(all_probs)    # (N, 5)
    all_labels = np.array(all_labels)   # (N, 5)

    aucs   = []
    auprcs = []
    class_aucs   = {}
    class_auprcs = {}

    for i, sc in enumerate(SUPERCLASSES):
        if all_labels[:, i].sum() > 0:
            auc   = roc_auc_score(all_labels[:, i], all_probs[:, i])
            auprc = average_precision_score(all_labels[:, i], all_probs[:, i])
            aucs.append(auc)
            auprcs.append(auprc)
            class_aucs[sc]   = auc
            class_auprcs[sc] = auprc

    macro_auc   = np.mean(aucs)
    macro_auprc = np.mean(auprcs)

    return total_loss / len(loader), macro_auc, macro_auprc, class_aucs, class_auprcs


num_epochs   = 20
best_val_auprc = 0.0

downstream_ckpt_dir = Path("checkpoints_tok1_downstream")
downstream_ckpt_dir.mkdir(parents=True, exist_ok=True)

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_auc, val_auprc, val_class_aucs, val_class_auprcs = evaluate(
        model, val_loader, criterion, device
    )

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss    : {train_loss:.4f}")
    print(f"  Val Loss      : {val_loss:.4f}")
    print(f"  Val Macro AUC : {val_auc:.4f}")
    print(f"  Val Macro AUPRC: {val_auprc:.4f}")
    for sc in SUPERCLASSES:
        if sc in val_class_aucs:
            print(f"    {sc}: AUC={val_class_aucs[sc]:.4f}  AUPRC={val_class_auprcs[sc]:.4f}")
    print("-" * 40)

    if val_auprc > best_val_auprc:
        best_val_auprc = val_auprc
        torch.save(model.state_dict(), downstream_ckpt_dir / "best.pt")
        print(f"  Saved new best model (val Macro AUPRC: {val_auprc:.4f})")

best_state = torch.load(downstream_ckpt_dir / "best.pt", map_location=device)
model.load_state_dict(best_state)
print(f"\nLoaded best model (val Macro AUPRC: {best_val_auprc:.4f})")

test_loss, test_auc, test_auprc, test_class_aucs, test_class_auprcs = evaluate(
    model, test_loader, criterion, device
)

print("\n========== TEST RESULTS ==========")
print(f"Test Loss       : {round(test_loss, 4)}")
print(f"Test Macro AUC  : {round(test_auc, 4)}")
print(f"Test Macro AUPRC: {round(test_auprc, 4)}")
print("\nPer-class results:")
for sc in SUPERCLASSES:
    if sc in test_class_aucs:
        print(f"  {sc}: AUC={round(test_class_aucs[sc], 4)}  AUPRC={round(test_class_auprcs[sc], 4)}")

Epoch 1/20
  Train Loss    : 0.3825
  Val Loss      : 0.3513
  Val Macro AUC : 0.8669
  Val Macro AUPRC: 0.6731
    NORM: AUC=0.9109  AUPRC=0.8771
    MI: AUC=0.8719  AUPRC=0.7042
    STTC: AUC=0.9022  AUPRC=0.7072
    CD: AUC=0.8758  AUPRC=0.7446
    HYP: AUC=0.7738  AUPRC=0.3325
----------------------------------------
  Saved new best model (val Macro AUPRC: 0.6731)
Epoch 2/20
  Train Loss    : 0.3276
  Val Loss      : 0.3406
  Val Macro AUC : 0.8753
  Val Macro AUPRC: 0.6950
    NORM: AUC=0.9161  AUPRC=0.8813
    MI: AUC=0.8822  AUPRC=0.7311
    STTC: AUC=0.9096  AUPRC=0.7314
    CD: AUC=0.8856  AUPRC=0.7736
    HYP: AUC=0.7830  AUPRC=0.3577
----------------------------------------
  Saved new best model (val Macro AUPRC: 0.6950)
Epoch 3/20
  Train Loss    : 0.3109
  Val Loss      : 0.3346
  Val Macro AUC : 0.8815
  Val Macro AUPRC: 0.7084
    NORM: AUC=0.9205  AUPRC=0.8868
    MI: AUC=0.8888  AUPRC=0.7432
    STTC: AUC=0.9120  AUPRC=0.7364
    CD: AUC=0.9009  AUPRC=0.7955
    HYP: